# Linear Algebra Deconvolution Simulation

Andrew E. Davidson aedavids@ucsc.edu 1/29/25 

Copyright (c) 2020-2023, Regents of the University of California All rights reserved.   https://polyformproject.org/licenses/noncommercial/1.0.0


CIBERSORTx is unable to deconvolve the serial dilution samples. Try relaxing the count & non-negative constraints. Solve using standard linear algebra

ref: linearAlgebraDeconvolution.ipynb
linearAlgebraDeconvolution.ipynb did not work. try using simulated data

In [1]:
import ipynbname

import numpy as np
import os
import pandas as pd

In [2]:
notebookName = ipynbname.name()
notebookPath = ipynbname.path()
notebookDir = os.path.dirname(notebookPath)

outDir = f'{notebookDir}/{notebookName}.out'

dataOut = f'{outDir}/data'
os.makedirs(dataOut, exist_ok=True) 
print(f'dataOut:\n{dataOut}')

# imgOut = f'{outDir}/img'
# os.makedirs(imgOut, exist_ok=True) 
# print(f'imgOut:\n{imgOut}')


dataOut:
/private/home/aedavids/extraCellularRNA/deconvolutionAnalysis/python/tempus/jupyterNotebooks/linearAlgebraDeconvolutionSimulation.out/data


## Generate Random Data

In [14]:
meaningOfLife = 42
np.random.seed(meaningOfLife)

numGenes = 15

def creatSignatureDataFrame( numGenes: int ) -> (pd.DataFrame, list[str]):
    '''
    TODO
    '''

    # multiply by a million so we have a strong signal with similar order of magnitude to
    # /private/groups/kimlab/aedavids/deconvolution/tempus/best/bestArithemetic10/bestArithemetic10Tempus.sh.out/
    # cibersortInputDir/arithmeticBiomarkers_T1.csv
    
    undiluted,control = np.random.rand(2, numGenes) * 1e6
    
    # print( f'undiluted\n {undiluted}')
    # print( f'\ncontrol\n {control}')
    
    geneIds = ['g' + str(i + 1) for i in range(numGenes) ]
    geneIds
    
    signatureDF = pd.DataFrame(
            data = np.array( [undiluted, control] ).transpose(),
            index = geneIds,
            columns = ['undiluted', 'control']
    )
    signatureDF.index.name = 'gene_id'

    return (signatureDF, geneIds)


signatureDF, geneIds = creatSignatureDataFrame( numGenes )
print(f'geneIds : {geneIds}')
print(f'signatureDF\n')
signatureDF

geneIds : ['g1', 'g2', 'g3', 'g4', 'g5', 'g6', 'g7', 'g8', 'g9', 'g10', 'g11', 'g12', 'g13', 'g14', 'g15']
signatureDF



,undiluted,control
gene_id,,
g1,374540.118847,183404.509853
g2,950714.306410,304242.242960
g3,731993.941811,524756.431632
g4,598658.484197,431945.018642
g5,156018.640442,291229.140198
g6,155994.520336,611852.894722
g7,58083.612168,139493.860652
g8,866176.145775,292144.648535
g9,601115.011743,366361.843294


In [6]:
# create mixture matrix

def createMixtureDF( 
        signatureDF : pd.DataFrame,
        dilutions : list[float]
) -> pd.DataFrame :
    '''
    TODO
    '''
    undilutedSeries = signatureDF.loc[:, "undiluted"]
    controlSeries   = signatureDF.loc[:, "control"]

    retDF = signatureDF.copy()
    
    for dilution in dilutions :
        sampleId = f'inSilco_{dilution}'
        c = controlSeries * (1 - dilution)
        d = undilutedSeries * dilution
        s = c + d
        retDF[ sampleId ] = s

    return retDF

dilutions = [1e-1, 1e-2, 1e-3, 1e-4, 1e-5, 1e-6]
mixtureDF = createMixtureDF( signatureDF,  dilutions)

print(f'mixtureDF\n')
mixtureDF

mixtureDF



,undiluted,control,inSilco_0.1,inSilco_0.01,inSilco_0.001,inSilco_0.0001,inSilco_1e-05,inSilco_1e-06
gene_id,,,,,,,,
g1,374540.118847,183404.509853,202518.070753,185315.865943,183595.645462,183423.623414,183406.421210,183404.700989
g2,950714.306410,304242.242960,368889.449305,310706.963594,304888.715023,304306.890166,304248.707680,304242.889432
g3,731993.941811,524756.431632,545480.182650,526828.806734,524963.669142,524777.155383,524758.504007,524756.638870
g4,598658.484197,431945.018642,448616.365198,433612.153298,432111.732108,431961.689989,431946.685777,431945.185356
g5,156018.640442,291229.140198,277708.090222,289877.035200,291093.929698,291215.619148,291227.788093,291229.004988
g6,155994.520336,611852.894722,566267.057284,607294.310979,611397.036348,611807.308885,611848.336139,611852.438864
g7,58083.612168,139493.860652,131352.835804,138679.758167,139412.450404,139485.719627,139493.046550,139493.779242
g8,866176.145775,292144.648535,349547.798259,297884.963508,292718.680032,292202.051685,292150.388850,292145.222567
g9,601115.011743,366361.843294,389837.160139,368709.374978,366596.596462,366385.318611,366364.190825,366362.078047


In [13]:
def solveForFractions( 
    signatureDF : pd.DataFrame,
    mixtureDF : pd.DataFrame,
) -> np.array :
    '''
    TODO
    '''
    # select numeric cols
    numericCols = ~signatureDF.columns.isin( ["gene_id"] )                                     
    sigNP = signatureDF.loc[:, numericCols].values
    
    # select numeric columns
    numericCols = ~mixtureDF.columns.isin( ["gene_id"] )                                     
    mixtureNP = mixtureDF.loc[:, numericCols].values

    print(f' sigNP.shape : {sigNP.shape} mixtureNP.shape : {mixtureNP.shape}')
    # 
    # Calculate the pseudo-inverse of the signature matrix
    signaturePinvNP = np.linalg.pinv( sigNP )
    # print(f"\n signaturePinvNP.shape : {signaturePinvNP.shape}\n{signaturePinvNP}")
    
    print("\n\n")
    
    FTranspose = np.matmul( signaturePinvNP, mixtureNP)
    # print(f"Matrix FTranspose.shape :{FTranspose.shape}")
    # print(FTranspose)
    
    # print(f"\n\nF.shape :{FTranspose.shape}")
    # print(np.transpose( FTranspose) )    
    

    FNP = FTranspose.transpose()
    return FNP

fractionsNP = solveForFractions( signatureDF, mixtureDF )
print(f'\n******************* fractionsNP.shape : {fractionsNP.shape}\n')
fractionsDF

 sigNP.shape : (15, 2) mixtureNP.shape : (15, 8)




******************* fractionsNP.shape : (8, 2)



array([[1.00000000e+00, 4.43320643e-17, 1.00000000e-01, 1.00000000e-02,
        1.00000000e-03, 1.00000000e-04, 1.00000000e-05, 1.00000000e-06],
       [9.24459074e-17, 1.00000000e+00, 9.00000000e-01, 9.90000000e-01,
        9.99000000e-01, 9.99900000e-01, 9.99990000e-01, 9.99999000e-01]])

In [18]:

FDF = pd.DataFrame(
    fractionsNP,
    index = mixtureDF.columns, 
    columns = ["UD", "Control", ]
)

FDF

,UD,Control
undiluted,1.000000e+00,9.244591e-17
control,4.433206e-17,1.000000e+00
inSilco_0.1,1.000000e-01,9.000000e-01
inSilco_0.01,1.000000e-02,9.900000e-01
inSilco_0.001,1.000000e-03,9.990000e-01
inSilco_0.0001,1.000000e-04,9.999000e-01
inSilco_1e-05,1.000000e-05,9.999900e-01
inSilco_1e-06,1.000000e-06,9.999990e-01


Index(['undiluted', 'control', 'inSilco_0.1', 'inSilco_0.01', 'inSilco_0.001',
       'inSilco_0.0001', 'inSilco_1e-05', 'inSilco_1e-06'],
      dtype='object')

In [ ]:
aedwip

## Solve for fractions matrix

In [ ]:

# select numeric columns
numericCols = ~mixtureDF.columns.isin( ["gene_id"] )

sampleIds= mixtureDF.columns[numericCols]
FDF = pd.DataFrame(
    FTranspose,
    columns = sampleIds, 
    index = ["UD", "Control", ]
)

FDF.transpose()

In [ ]:
FDF = FDF.transpose()
FDF

In [ ]:
FDF = FDF.round(decimals=2)
FDF

In [ ]:
byRow = 1
FDF['rowSum'] =FDF.loc[:, ['UD', 'Control']].sum(axis=byRow)
FDF

In [ ]:
FDF['%UD'] =  FDF['UD'] / FDF['rowSum'] * 100.0 
# print()
# print( FDF['Control'] / FDF['rowSum'] * 100.0)
FDF['%Control'] =  FDF['Control'] / FDF['rowSum'] * 100.0 

orderedIdx = ['SLDK3_T1_100_S1_L007', 'SLDK3_T1_1K_S4_L007', 'SLDK3_T1_10K_S3_L007',
             'SLDK3_T1_100K_S2_L007', 'SLDK3_T1_1M_S5_L007', 
             'SLDK3_T1_Control_S6_L007', 'SLDK3_T1_UD_S7_L007']

FDF.loc[orderedIdx, :]